In [5]:
# Import libraries
from pathlib import Path
import json
import pandas as pd

In [6]:
# Set up paths
scenario_path = Path().resolve()
annotated_outputs_path = scenario_path.parent.parent / "annotated_outputs" / "moral_comp_exp4"

In [7]:
scenarios_to_read = ["endowment.json", "garden.json", "servicedog.json"]


def parse_choice_1(scenario_name, variant_id):
    """Parse a choice-1 annotation from the scenario's output folder."""
    scenario_output_path = annotated_outputs_path / scenario_name
    output_file = scenario_output_path / f"{scenario_name}_{variant_id}_choice_1.json"

    with output_file.open("r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    node_records = [record for record in records if "node" in record]

    action_record = next(
        record for record in node_records
        if record["node"]["kind"] == "action_choice"
    )

    action_value = next(
        link["link"]["value"]
        for link in action_record["links"]
        if link["link"]["kind"] == "v-link"
    )

    annotator_entities = [
        record["node"]["label"]
        for record in node_records
        if record["node"]["kind"] == "being"
    ]

    event_records = [
        record for record in node_records
        if record["node"]["kind"] == "event"
    ]

    annotator_outcomes = [
        record["node"]["label"]
        for record in event_records
    ]

    entities_with_outcome_links = {
        link["to_node"]
        for record in event_records
        for link in record["links"]
        if link["link"]["kind"] == "utility"
    }

    annotator_entities_without_outcomes = [
        entity for entity in annotator_entities
        if entity not in entities_with_outcome_links
    ]

    return (
        action_record["node"]["label"],
        action_value,
        annotator_entities,
        annotator_outcomes,
        annotator_entities_without_outcomes,
    )

review_rows = []

for scenario_filename in scenarios_to_read:
    scenario_name = Path(scenario_filename).stem
    scenario_file = scenario_path / scenario_filename

    with scenario_file.open("r", encoding="utf-8") as f:
        variants = json.load(f)

    for variant in variants:
        action, action_value, annotator_entities, annotator_outcomes, entities_without_outcomes = parse_choice_1(
            scenario_name,
            variant["id"],
        )

        review_rows.append({
            "id": variant["id"],
            "scenario_title": variant["scenario_title"],
            "deont_level": variant["deontology_level"],
            "util_level": variant["utility_level"],
            "action": action,
            "action_value": action_value,
            "scenario_text": variant["text"],
            "annotator_entities": annotator_entities,
            "annotator_entities_without_outcomes": entities_without_outcomes,
            "annotator_outcomes": annotator_outcomes,
        })

review_df = pd.DataFrame(review_rows)

for _, row in review_df.iterrows():
    print("=" * 100)
    print(json.dumps(row.to_dict(), ensure_ascii=False, indent=2))

{
  "id": 1,
  "scenario_title": "Endowment",
  "deont_level": "1",
  "util_level": "3",
  "action": "accept the donation that requires the foundation to be named after the murderer",
  "action_value": "-79.0",
  "scenario_text": "After a long-term imprisonment and while awaiting a parole decision, a convicted murderer wants to organise a foundation for young victims of crime. I am the president of the agency who decides about the foundation. The founder would donate 250,000 dollars to the foundation. However, he asks that the foundation would be named after himself, helping clear his name. The family of his victim opposes such a foundation and considers it highly upsetting. If I accept the donation that requires the foundation to be named after the murderer, crime victims will benefit from the 250,000 dollars. If I do not do this, the money will be lost.",
  "annotator_entities": [
    "i",
    "the convicted murderer / prospective founder and donor",
    "the victim's family members 